# 🎮 Tetris-RL — Google Colab Training Notebook
Single-file notebook to train PPO on Game Boy Tetris via **PyBoy + Gymnasium + Stable-Baselines3**.

* Repo: `https://github.com/blackccpie/tetris-rl` `feat/ubuntu-24-portability` (6 actions `LEFT/RIGHT/DOWN/A/B/PASS`, `freq12`, `TetrisCNN 18×10`, PBRS `α=0.1` `-0.5/+0.75/-0.8/-0.35`, `ent 0.02`)
* Runtime: any Colab (CPU `~280 fps n_envs4`, T4 GPU `~300 fps` — emulator-bound, GPU not required)
* ROM: `roms/tetris.gb` **not tracked** — you must provide a legal dump (32 KiB). `states/init.state` is already in repo (PyBoy 2.7.0, press START after boot).

> **Steps:** 1) Run cells top→bottom  2) Upload ROM when prompted  3) Train → checkpoints in `models/` → download `.zip`


In [ ]:
# @title 1 — System deps (libsdl2 runtime, Colab Ubuntu)
# PyBoy needs SDL2 at runtime; Ubuntu 24.04 (Colab) already has it, but ensure:
import subprocess, sys
try:
    import shutil
    has_sdl = shutil.which("apt") is not None
    print("apt available, installing libsdl2-2.0-0 ...")
    # !apt -qq update && apt -qq install -y libsdl2-2.0-0 > /dev/null
    # Colab often forbids apt update in some images — try, ignore failure:
    subprocess.run(["apt","update","-qq"], check=False)
    subprocess.run(["apt","install","-y","-qq","libsdl2-2.0-0"], check=False)
except Exception as e:
    print("apt skip:", e)

# Verify GPU
try:
    import torch
    print("torch", torch.__version__, "cuda", torch.cuda.is_available())
    if torch.cuda.is_available():
        print(torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))
except Exception as e:
    print("torch check:", e)


In [ ]:
# @title 2 — Clone repo + Python deps (uv or pip)
import os, pathlib, subprocess, sys, textwrap

REPO = "https://github.com/blackccpie/tetris-rl.git"
BRANCH = "feat/ubuntu-24-portability"  # has 6-action, freq12, ent0.02, lipo fixes
WORK = "/content/tetris-rl"

if not pathlib.Path(WORK).exists():
    print(f"Cloning {REPO} @ {BRANCH} ...")
    subprocess.run(["git","clone","--depth","1","--branch",BRANCH, REPO, WORK], check=True)
else:
    print("Repo exists, pulling ...")
    subprocess.run(["git","-C",WORK,"fetch","--depth","1","origin",BRANCH], check=False)
    subprocess.run(["git","-C",WORK,"checkout",BRANCH], check=False)
    subprocess.run(["git","-C",WORK,"pull","--ff-only"], check=False)

print("Repo at", WORK, os.listdir(WORK)[:6])

# Install deps — Colab already has torch; use pip for rest
# Option A: uv sync (if uv installed) — comment out pip below
# !pip install -q uv && uv --directory {WORK} sync --extra dev
#
# Option B: pip (compatible with Colab py3.10-3.12)
# pyproject requires python>=3.12 but Colab is 3.10/3.11 — pip --no-deps bypass, then install deps manually:
print("Installing python deps via pip ...")
subprocess.run([sys.executable,"-m","pip","install","-q",
    "gymnasium","numpy","pyboy","stable-baselines3","tqdm","tensorboard",
    "pillow"
], check=False)
# Ensure torch present (Colab preinstalled). If you want legacy sm_52 pin: torch==2.6.0+cu124
try:
    import torch
    print("torch kept:", torch.__version__)
except ImportError:
    subprocess.run([sys.executable,"-m","pip","install","-q","torch==2.6.0"], check=False)

%cd {WORK}
print("workdir", os.getcwd())


In [ ]:
# @title 3 — Upload ROM (legal dump) → roms/tetris.gb  [+ verify init.state]
import pathlib, os
from google.colab import files

ROM_DST = pathlib.Path("roms/tetris.gb")
INIT_SRC = pathlib.Path("states/init.state")
ROM_DST.parent.mkdir(parents=True, exist_ok=True)

if ROM_DST.exists() and ROM_DST.stat().st_size >= 32768:
    print(f"ROM exists: {ROM_DST} {ROM_DST.stat().st_size} bytes")
else:
    print("Upload your legal Game Boy Tetris ROM (32 KiB, file must be named tetris.gb)")
    print("In file picker, select your roms/tetris.gb dump — will be saved to", ROM_DST)
    uploaded = files.upload()  # dict {filename: bytes}
    # files.upload returns keys as uploaded name; save first file to ROM_DST
    if uploaded:
        name, data = next(iter(uploaded.items()))
        ROM_DST.write_bytes(data)
        print(f"Saved {name} ({len(data)} bytes) → {ROM_DST}")
    else:
        print("No file uploaded!")

print(f"ROM: {ROM_DST} exists={ROM_DST.exists()} size={ROM_DST.stat().st_size if ROM_DST.exists() else 'missing'}")
print(f"Init: {INIT_SRC} exists={INIT_SRC.exists()} size={INIT_SRC.stat().st_size if INIT_SRC.exists() else 'missing'}  (regen: press START after boot if version mismatch)")
if ROM_DST.exists():
    # quick magic check
    data = ROM_DST.read_bytes()[:16]
    print("ROM header hex:", data.hex()[:40], "... title bytes", data[0x34-0x100:0x34-0x100+10] if len(data)>0 else "")

# Ensure models/ exists (train.py creates, but pre-create)
pathlib.Path("models").mkdir(parents=True, exist_ok=True)
print("models/", list(pathlib.Path("models").glob("*"))[:5])


In [ ]:
# @title 4 — Smoke test env (6 actions, freq12, window null, 500 steps 145 fps)
import sys
sys.path.insert(0, ".")
from tetris_env import tetris_env
from pathlib import Path
assert Path("roms/tetris.gb").exists(), "ROM missing — re-run upload cell"
assert Path("states/init.state").exists(), "init.state missing"

env = tetris_env(gb_path="roms/tetris.gb", window="null", shaped_alpha=0.1)
print("action_space", env.action_space, "n", env.action_space.n)
from tetris_env import action_names
print([(i, action_names[a]) for i,a in enumerate(env.valid_actions)], "freq", env.action_freq, "weights", env.shaped_weights)
import time
obs,_ = env.reset(seed=42)
t0=time.time()
for _ in range(500):
    obs,_,term,_,_=env.step(env.action_space.sample())
    if term: obs,_=env.reset(seed=42)
print(f"500 steps {time.time()-t0:.2f}s fps {500/(time.time()-t0):.1f}")
env.close()
print("✓ env ok — 6 actions LEFT/RIGHT/DOWN/A/B/PASS, UP removed")


In [ ]:
# @title 5 — Train PPO (configurable) — defaults 1.6M = 200×4×2048, 5M = 600×4×2048
# Tip: start small (sessions 10) to verify, then 200 or 600. Use Runtime → Interrupt execution to checkpoint (CTRL+C saves).
import subprocess, pathlib, torch

# ---- edit these ----
SESSIONS = 200      # 200→1.6M  600→5M  1200→10M  (×runs×steps×n_envs)
RUNS = 4
STEPS = 2048
N_ENVS = 4          # 1 DummyVecEnv, >1 SubprocVecEnv (4→280 fps cpu, T4 ~300)
POLICY = "CnnPolicy"  # CnnPolicy TetrisCNN (18×10) recommended, MlpPolicy fallback
SHAPED_ALPHA = 0.1  # 0.0 legacy sparse, 0.1 hybrid
FREQ = 12           # must match play.py — 12 fine rotation, 24 legacy
ENT_COEF = 0.02     # 0.02 prevents rotation collapse (0.01 collapsed at 8k)
DEVICE = "auto"     # auto|cpu|cuda — auto→cpu on Colab CPU, cuda on T4
WINDOW = "null"     # null for training (headless)
CHECKPOINT_FREQ = 10
TENSORBOARD = ""    # e.g. "logs/colab" or "" to disable
MODEL_NAME = "models/tetris_ppo_model"  # arch old 7-action zips → fresh start (see archive7/)

args = [
    "python","train.py",
    "--sessions",str(SESSIONS),
    "--runs",str(RUNS),
    "--steps",str(STEPS),
    "--n-envs",str(N_ENVS),
    "--policy",POLICY,
    "--shaped-alpha",str(SHAPED_ALPHA),
    "--freq",str(FREQ),
    "--ent-coef",str(ENT_COEF),
    "--device",DEVICE,
    "--window",WINDOW,
    "--checkpoint-freq",str(CHECKPOINT_FREQ),
    "--model-name",MODEL_NAME,
]
if TENSORBOARD:
    args += ["--tensorboard", TENSORBOARD]

print("Running:", " ".join(args))
print("Device torch cuda", torch.cuda.is_available(), "->", "cuda" if DEVICE=="cuda" or (DEVICE=="auto" and torch.cuda.is_available()) else "cpu")
print("Total env steps ≈", SESSIONS*RUNS*STEPS*N_ENVS, f"({SESSIONS}×{RUNS}×{STEPS}×{N_ENVS})")
# Stream output (tqdm + PPO logs)
proc = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print("Exit code", proc.returncode)
print("Models:", list(pathlib.Path("models").glob("*.zip"))[:10])


In [ ]:
# @title 5b — Segmented resumable (Option A) — 5M as 10×60 (~500k each), survives disconnects
# Alternative to single Train cell — splits into segments, each resumes from models/tetris_ppo_model.zip
TOTAL = 600
PER_SEGMENT = 60
N_ENVS_SEG = 4
# Run: bash scripts/train_segments.sh --total 600 --per-segment 60 --n-envs 4 --device auto
import subprocess, pathlib
import shlex
cmd = ["bash","scripts/train_segments.sh","--total",str(TOTAL),"--per-segment",str(PER_SEGMENT),"--n-envs",str(N_ENVS_SEG),"--device","auto"]
print("Running:", " ".join(shlex.quote(c) for c in cmd))
# Uncomment to run:
# subprocess.run(cmd, check=False)
print("To run, uncomment last line or execute in terminal cell: !bash scripts/train_segments.sh --total 600 --per-segment 60")
print("Checkpoints:", list(pathlib.Path("models").glob("*.zip"))[:10])


In [ ]:
# @title 6 — Evaluate checkpoints (action histogram, deterministic vs stochastic, rotation check)
import pathlib, collections, numpy as np, torch, sys
sys.path.insert(0, ".")
from stable_baselines3 import PPO
from train import TetrisCNN
from tetris_env import tetris_env

CKPT = "models/tetris_ppo_model.zip"  # or models/tetris_ppo_model_ckpt_0010.zip
FREQ_EVAL = 12
SHAPED_ALPHA_EVAL = 0.1

if not pathlib.Path(CKPT).exists():
    print(f"Checkpoint {CKPT} missing — train first or list models/")
    print(list(pathlib.Path("models").glob("*.zip")))
else:
    env = tetris_env(gb_path="roms/tetris.gb", window="null", shaped_alpha=SHAPED_ALPHA_EVAL, action_freq=FREQ_EVAL)
    m = PPO.load(CKPT, env=env, device="cpu", custom_objects={"TetrisCNN": TetrisCNN})
    print("Loaded", CKPT, m.policy.__class__.__name__, "timesteps", m.num_timesteps, "action_space", m.action_space)

    for det in [True, False]:
        obs,_ = env.reset(seed=42)
        c = collections.Counter()
        for _ in range(500):
            a,_ = m.predict(obs, deterministic=det)
            c[int(a)]+=1
            obs,_,term,_,_=env.step(int(a))
            if term: obs,_=env.reset(seed=np.random.randint(0,100000))
        names = {i: ("LEFT","RIGHT","DOWN","A","B","PASS")[i] for i in range(6)}
        print(("deterministic" if det else "stochastic"), dict(c), "rotation", c[3]+c[4], "/500", {k:names[k] for k in c})
        # Tip: deterministic should show A/B if trained well; stochastic shows exploration
    # Logits at fixed state
    obs,_ = env.reset(seed=42)
    with torch.no_grad():
        probs = m.policy.get_distribution(torch.as_tensor(obs).unsqueeze(0)).distribution.probs
        print("probs at seed42", probs.numpy().round(3), "entropy", float(-(probs*probs.log()).sum()))
    env.close()

    # Quick headless rollout scores
    env2 = tetris_env(gb_path="roms/tetris.gb", window="null", shaped_alpha=SHAPED_ALPHA_EVAL, action_freq=FREQ_EVAL)
    m2 = PPO.load(CKPT, env=env2, device="cpu", custom_objects={"TetrisCNN": TetrisCNN})
    for _ in range(3):
        obs,_=env2.reset(seed=np.random.randint(0,100000))
        steps=0; term=False
        while not term and steps<2000:
            a,_=m2.predict(obs, deterministic=False)  # stochastic to see rotation
            obs,_,term,_,_=env2.step(int(a))
            steps+=1
        print(f"Rollout steps {steps} score {env2.get_game_score()}")
    env2.close()


In [ ]:
# @title 7 — TensorBoard (if TENSORBOARD logs/ used) + download model
import pathlib, subprocess, sys
# TensorBoard inline (Colab): load extension then %tensorboard
try:
    get_ipython().run_line_magic("load_ext","tensorboard")
    print("Run: %tensorboard --logdir logs  (then set TENSORBOARD='logs' in Train cell)")
except Exception as e:
    print(e)

# List checkpoints
for p in sorted(pathlib.Path("models").glob("*.zip")):
    print(f"{p} {p.stat().st_size/1024/1024:.1f} MiB")

# Download best model to local machine
from google.colab import files
import pathlib
ZIP = pathlib.Path("models/tetris_ppo_model.zip")
if ZIP.exists():
    print(f"Downloading {ZIP} ... (browser prompt)")
    # files.download will trigger download; comment out if file too large for browser
    # files.download(str(ZIP))
    print(f"To download manually: files.download('{ZIP}')")
else:
    print("No model yet — train first")


In [ ]:
# @title 8 — Play with SDL2 window (local) or headless eval — match train freq/alpha
# In Colab you cannot show SDL2 window — use --window null for headless.
# For local machine: python play.py --window SDL2 --model models/tetris_ppo_model --freq 12 --shaped-alpha 0.1
import subprocess
# Headless demo 2 runs:
subprocess.run(["python","play.py","--window","null","--runs","2","--freq","12","--shaped-alpha","0.1","--model","models/tetris_ppo_model","--deterministic"], check=False)
# Stochastic (shows more rotation if ent low):
# subprocess.run(["python","play.py","--window","null","--runs","2","--freq","12","--no-deterministic"], check=False)
